# Auto-Label Data with Event Triggers

Learn how to automatically label sensor data using hardware event triggers (button press, GPIO, PWM):
- Detect button press/release events
- Automatically label gesture segments
- Create manifest and upload to MPLAB ML

**Use Case:** You're collecting gesture, activity, or event data with a physical trigger (button, switch, or sensor) that marks when events occur.

## Why Event-Triggered Labeling?

**Manual labeling is tedious:**
- Reviewing long recordings
- Finding exact start/end times
- Prone to human error

**Event triggers automate this:**
- Press button → gesture starts
- Release button → gesture ends
- Labels applied automatically

**Common scenarios:**
- 🎯 Gesture recognition (button pressed during gesture)
- 🏃 Activity tracking (start/stop activities)
- ⚡ Anomaly detection (trigger on fault events)
- 🔧 Machine monitoring (trigger on specific operations)

---

## Example Dataset

**Sensor:** 6-axis IMU (accelerometer + gyroscope)
**Gesture:** "wave" motion
**Trigger:** Button pressed during gesture, released when idle
**Data:** 950 samples (~9.5 seconds @ 100 Hz)
**Events:** 3 wave gestures with idle periods between

---

## 1. Setup and Load Data

### Install Dependencies

In [ ]:
# Install MPLAB ML SDK
!pip install mplabml --quiet

print("✓ MPLAB ML SDK installed")

### Import Libraries

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from getpass import getpass
from mplabml import Client

print("✓ Libraries imported")

### Load Sample Dataset

**Two options for loading data:**

In [ ]:
# Option 1: Load from GitHub (recommended - works everywhere)
GITHUB_URL = 'https://raw.githubusercontent.com/MicrochipTech/mplabml-sdk-examples/main/datasets/gesture_sample.csv'

try:
    df = pd.read_csv(GITHUB_URL)
    print("✓ Loaded from GitHub")
except Exception as e:
    print(f"Could not load from GitHub: {e}")
    print("\nTrying local file...")
    
    # Option 2: Load local file
    # For Colab: Upload to /content/
    # For local Jupyter: Use ../datasets/gesture_sample.csv
    
    try:
        df = pd.read_csv('/content/gesture_sample.csv')
        print("✓ Loaded from /content/")
    except:
        df = pd.read_csv('../datasets/gesture_sample.csv')
        print("✓ Loaded from ../datasets/")

print(f"\nDataset loaded: {len(df):,} samples")
print(f"Columns: {list(df.columns)}")

---
## 2. Understand the Data

### What's in the Dataset?

**Sensor Data (6 columns):**
- AccelerometerX, Y, Z - Motion in 3 axes (mg)
- GyroscopeX, Y, Z - Rotation in 3 axes (deg/s)

**Trigger (1 column):**
- 0 = Button released (idle)
- 1 = Button pressed (gesture happening)

### Inspect the Data

In [ ]:
# Basic statistics
print("Dataset Overview:")
print("=" * 60)
print(f"Total samples: {len(df):,}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nData types:\n{df.dtypes}")

# Trigger statistics
print("\n" + "=" * 60)
print("Trigger Signal:")
print("=" * 60)
trigger_counts = df['Trigger'].value_counts().sort_index()
for value, count in trigger_counts.items():
    state = "released (idle)" if value == 0 else "pressed (gesture)"
    pct = 100.0 * count / len(df)
    print(f"  {value} ({state:20s}): {count:4d} samples ({pct:5.1f}%)")

# Show first few rows
print("\nFirst 10 samples:")
print(df.head(10))

### Visualize Sensor Data and Trigger

**This plot shows:**
- **Top:** Accelerometer signals (motion)
- **Middle:** Gyroscope signals (rotation)
- **Bottom:** Trigger signal (button state)

**What to look for:**
- When Trigger = 1 (button pressed), you should see IMU changes
- When Trigger = 0 (button released), IMU should be relatively stable

In [ ]:
def plot_imu_with_trigger(df, title="Sensor Data with Button Trigger"):
    """
    Plot IMU data and trigger signal to visualize event timing.
    """
    fig, axes = plt.subplots(3, 1, figsize=(14, 10))
    
    # Create sample index for x-axis
    samples = range(len(df))
    
    # Plot 1: Accelerometer
    axes[0].plot(samples, df['AccelerometerX'], label='X', alpha=0.7)
    axes[0].plot(samples, df['AccelerometerY'], label='Y', alpha=0.7)
    axes[0].plot(samples, df['AccelerometerZ'], label='Z', alpha=0.7)
    axes[0].set_ylabel('Accelerometer (mg)')
    axes[0].legend(loc='upper right')
    axes[0].set_title(title)
    axes[0].grid(True, alpha=0.3)
    
    # Plot 2: Gyroscope
    axes[1].plot(samples, df['GyroscopeX'], label='X', alpha=0.7)
    axes[1].plot(samples, df['GyroscopeY'], label='Y', alpha=0.7)
    axes[1].plot(samples, df['GyroscopeZ'], label='Z', alpha=0.7)
    axes[1].set_ylabel('Gyroscope (deg/s)')
    axes[1].legend(loc='upper right')
    axes[1].grid(True, alpha=0.3)
    
    # Plot 3: Trigger
    axes[2].plot(samples, df['Trigger'], color='red', linewidth=2)
    axes[2].fill_between(samples, 0, df['Trigger'], alpha=0.3, color='red')
    axes[2].set_ylabel('Trigger (Button)')
    axes[2].set_xlabel('Sample Index')
    axes[2].set_ylim([-0.1, 1.1])
    axes[2].set_yticks([0, 1])
    axes[2].set_yticklabels(['Released', 'Pressed'])
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Visualize the data
plot_imu_with_trigger(df)

print("\n💡 Notice: IMU signals change when button is pressed (Trigger=1)")
print("   This correlation lets us automatically label gesture segments!")

---
## 3. Detect Button Events

### How Event Detection Works

**We look for "edges" in the trigger signal:**

**Rising Edge (0 → 1):**
- Button pressed
- Gesture START

**Falling Edge (1 → 0):**
- Button released
- Gesture END

**Between edges = labeled segment**

### Implement Edge Detection

In [ ]:
def detect_button_edges(trigger_signal):
    """
    Detect rising and falling edges in trigger signal.
    
    Rising edge (0→1): Button pressed, gesture starts
    Falling edge (1→0): Button released, gesture ends
    
    Returns:
        rising_edges: Sample indices where button was pressed
        falling_edges: Sample indices where button was released
    """
    # Convert to numpy array
    trigger = np.array(trigger_signal)
    
    # Calculate difference between consecutive samples
    # diff[i] = trigger[i] - trigger[i-1]
    diff = np.diff(trigger, prepend=0)
    
    # Rising edge: diff = +1 (went from 0 to 1)
    rising_edges = np.where(diff == 1)[0]
    
    # Falling edge: diff = -1 (went from 1 to 0)
    falling_edges = np.where(diff == -1)[0]
    
    return rising_edges, falling_edges

# Detect events in our data
rising_edges, falling_edges = detect_button_edges(df['Trigger'])

print("Button Event Detection:")
print("=" * 60)
print(f"Rising edges (button press):   {len(rising_edges)} events")
print(f"Falling edges (button release): {len(falling_edges)} events")

print("\nDetailed Events:")
print("-" * 60)
for i, (start, end) in enumerate(zip(rising_edges, falling_edges), 1):
    duration = end - start
    print(f"Event {i}:")
    print(f"  Button pressed at sample:  {start}")
    print(f"  Button released at sample: {end}")
    print(f"  Duration: {duration} samples\n")

### Visualize Detected Events

Let's mark the detected edges on our plot:

In [ ]:
def plot_with_event_markers(df, rising_edges, falling_edges):
    """
    Plot data with event markers showing where button was pressed/released.
    """
    fig, axes = plt.subplots(3, 1, figsize=(14, 10))
    samples = range(len(df))
    
    # Plot accelerometer with event markers
    axes[0].plot(samples, df['AccelerometerX'], label='X', alpha=0.7)
    axes[0].plot(samples, df['AccelerometerY'], label='Y', alpha=0.7)
    axes[0].plot(samples, df['AccelerometerZ'], label='Z', alpha=0.7)
    
    # Mark rising edges (START)
    for edge in rising_edges:
        axes[0].axvline(x=edge, color='green', linestyle='--', alpha=0.7, linewidth=2)
    
    # Mark falling edges (END)
    for edge in falling_edges:
        axes[0].axvline(x=edge, color='red', linestyle='--', alpha=0.7, linewidth=2)
    
    axes[0].set_ylabel('Accelerometer (mg)')
    axes[0].legend(loc='upper right')
    axes[0].set_title('Sensor Data with Detected Events')
    axes[0].grid(True, alpha=0.3)
    
    # Plot gyroscope with event markers
    axes[1].plot(samples, df['GyroscopeX'], label='X', alpha=0.7)
    axes[1].plot(samples, df['GyroscopeY'], label='Y', alpha=0.7)
    axes[1].plot(samples, df['GyroscopeZ'], label='Z', alpha=0.7)
    
    for edge in rising_edges:
        axes[1].axvline(x=edge, color='green', linestyle='--', alpha=0.7, linewidth=2)
    for edge in falling_edges:
        axes[1].axvline(x=edge, color='red', linestyle='--', alpha=0.7, linewidth=2)
    
    axes[1].set_ylabel('Gyroscope (deg/s)')
    axes[1].legend(loc='upper right')
    axes[1].grid(True, alpha=0.3)
    
    # Plot trigger with shaded regions
    axes[2].plot(samples, df['Trigger'], color='red', linewidth=2)
    
    # Shade labeled regions
    for start, end in zip(rising_edges, falling_edges):
        axes[2].axvspan(start, end, alpha=0.3, color='green', label='Labeled Gesture' if start == rising_edges[0] else '')
    
    axes[2].set_ylabel('Trigger')
    axes[2].set_xlabel('Sample Index')
    axes[2].set_ylim([-0.1, 1.1])
    axes[2].set_yticks([0, 1])
    axes[2].set_yticklabels(['Released', 'Pressed'])
    axes[2].grid(True, alpha=0.3)
    axes[2].legend(loc='upper right')
    
    # Add legend for markers
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], color='green', linestyle='--', linewidth=2, label='START (button press)'),
        Line2D([0], [0], color='red', linestyle='--', linewidth=2, label='END (button release)')
    ]
    axes[0].legend(handles=legend_elements + axes[0].get_legend_handles_labels()[0], loc='upper right')
    
    plt.tight_layout()
    plt.show()

plot_with_event_markers(df, rising_edges, falling_edges)

print("\n💡 Green lines = Gesture START (button press)")
print("   Red lines = Gesture END (button release)")
print("   Shaded regions = Automatically labeled as 'wave' gesture")

---
## 4. Auto-Label the Data

### Labeling Strategy

**Label Assignment:**
- **Between events (button pressed)** → Label as "wave"
- **Outside events (button released)** → Label as "idle" or "unknown"

**Why this works:**
- Hardware trigger provides ground truth timing
- No manual review needed
- Scalable to thousands of samples

### Implement Auto-Labeling

In [ ]:
def auto_label_from_events(df, rising_edges, falling_edges, 
                           gesture_label='wave', idle_label='idle'):
    """
    Automatically label data based on button events.
    
    Args:
        df: DataFrame with sensor data and Trigger column
        rising_edges: Sample indices where button was pressed
        falling_edges: Sample indices where button was released
        gesture_label: Label for data between button press/release
        idle_label: Label for data outside button events
    
    Returns:
        DataFrame with added 'label' column
    """
    df = df.copy()
    
    # Initialize all samples as idle
    df['label'] = idle_label
    
    # Label each segment between button press and release
    segments_labeled = 0
    
    for start_idx, end_idx in zip(rising_edges, falling_edges):
        # Label the segment as gesture
        df.loc[start_idx:end_idx, 'label'] = gesture_label
        segments_labeled += 1
    
    print(f"Auto-Labeling Complete:")
    print("=" * 60)
    print(f"  Segments labeled: {segments_labeled}")
    print(f"  Total samples: {len(df):,}")
    
    # Show label distribution
    print(f"\nLabel Distribution:")
    label_counts = df['label'].value_counts()
    for label, count in label_counts.items():
        pct = 100.0 * count / len(df)
        print(f"  {label:10s}: {count:4d} samples ({pct:5.1f}%)")
    
    return df

# Apply auto-labeling
df_labeled = auto_label_from_events(df, rising_edges, falling_edges)

print("\n✓ Data automatically labeled!")
print("\nFirst 10 labeled samples:")
print(df_labeled[['AccelerometerX', 'AccelerometerY', 'AccelerometerZ', 'Trigger', 'label']].head(10))

### Visualize Labeled Data

Let's verify the labels are correct:

In [ ]:
def plot_labeled_data(df_labeled):
    """
    Plot sensor data color-coded by label.
    """
    fig, ax = plt.subplots(figsize=(14, 6))
    
    # Create sample index
    samples = range(len(df_labeled))
    
    # Plot accelerometer X-axis, colored by label
    # (Using just X-axis to keep plot clean)
    for label in df_labeled['label'].unique():
        mask = df_labeled['label'] == label
        color = 'green' if label == 'wave' else 'gray'
        alpha = 0.8 if label == 'wave' else 0.3
        ax.scatter(np.array(samples)[mask], df_labeled.loc[mask, 'AccelerometerX'], 
                  c=color, alpha=alpha, s=10, label=label)
    
    ax.set_xlabel('Sample Index')
    ax.set_ylabel('AccelerometerX (mg)')
    ax.set_title('Labeled Data (AccelerometerX shown)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

plot_labeled_data(df_labeled)

print("\n💡 Green = 'wave' gesture (button pressed)")
print("   Gray = 'idle' (button released)")

---
## 5. Prepare for MPLAB ML

### Create Project Files

MPLAB ML needs:
1. **CSV file** - Sensor data (we'll keep the Trigger column for verification)
2. **Manifest file (.dclproj)** - Label definitions and segments

**Note:** We keep the Trigger column in the CSV so you can verify in MPLAB ML that:
- Labels align correctly with button events
- Segments match the trigger timing
- Event detection worked as expected

This makes debugging and verification much easier!

### Save CSV File

In [ ]:
# Create output directory
OUTPUT_DIR = 'event_labeled_project'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save CSV (keep Trigger column for verification in MPLAB ML)
# Note: We exclude the 'label' column since MPLAB ML uses segments instead
columns_to_save = ['AccelerometerX', 'AccelerometerY', 'AccelerometerZ', 
                   'GyroscopeX', 'GyroscopeY', 'GyroscopeZ', 'Trigger']
csv_filename = 'gesture_data.csv'
csv_path = os.path.join(OUTPUT_DIR, csv_filename)

df_labeled[columns_to_save].to_csv(csv_path, index=False)

print(f"✓ Saved CSV: {csv_path}")
print(f"  Columns: {columns_to_save}")
print(f"  Size: {os.path.getsize(csv_path) / 1024:.1f} KB")
print(f"\n💡 Trigger column included for verification in MPLAB ML")

### Create Segments from Labels

**Segments tell MPLAB ML:**
- Which label applies
- Which samples (start to end)

**Example:**
```json
{
  "name": "Label",
  "value": "wave",
  "start": 200,
  "end": 349
}
```
= Samples 200-349 are labeled "wave"

In [ ]:
def create_segments_from_labels(df_labeled, label_col='label'):
    """
    Create segment definitions from labeled data.
    
    Detects where labels change and creates segment for each continuous region.
    
    Returns:
        List of segment dictionaries in MPLAB ML format
    """
    segments = []
    
    # Find where label changes
    df_labeled = df_labeled.reset_index(drop=True)
    label_changes = df_labeled[label_col].ne(df_labeled[label_col].shift())
    change_indices = label_changes[label_changes].index.tolist()
    
    # Add start and end
    change_indices = [0] + change_indices + [len(df_labeled)]
    
    # Create segments
    for i in range(len(change_indices) - 1):
        start_idx = change_indices[i]
        end_idx = change_indices[i + 1] - 1
        label = df_labeled.loc[start_idx, label_col]
        
        segment = {
            "name": "Label",
            "value": str(label),  # Must be string
            "start": int(start_idx),
            "end": int(end_idx)
        }
        segments.append(segment)
    
    return segments

# Create segments
segments = create_segments_from_labels(df_labeled)

print(f"Created {len(segments)} segment(s):")
print("=" * 60)
for i, seg in enumerate(segments, 1):
    duration = seg['end'] - seg['start'] + 1
    print(f"{i}. Label: {seg['value']:10s} | Samples {seg['start']:4d}-{seg['end']:4d} ({duration:4d} samples)")

# Verify segments match labels
total_segment_samples = sum(seg['end'] - seg['start'] + 1 for seg in segments)
print(f"\n✓ Verification: {total_segment_samples} segment samples = {len(df_labeled)} total samples")

### Create Manifest File

**Manifest (.dclproj) links the CSV to the segments:**

In [ ]:
# Create manifest structure
manifest = [{
    "file_name": csv_filename,
    "metadata": [
        {"name": "example_type", "value": "event_triggered_labeling"},
        {"name": "gesture", "value": "wave"},
        {"name": "sensor", "value": "6axis_IMU"},
        {"name": "sampling_rate_hz", "value": "100"},
        {"name": "description", "value": "auto_labeled_with_button_events"}
    ],
    "sessions": [{
        "session_name": "Session_1",
        "segments": segments
    }]
}]

# Save manifest
manifest_filename = 'event_labeled_project.dclproj'
manifest_path = os.path.join(OUTPUT_DIR, manifest_filename)
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f"✓ Saved manifest: {manifest_path}")
print(f"\nManifest preview (first 500 chars):")
print(json.dumps(manifest, indent=2)[:500] + "...")

### Verify Project Structure

In [ ]:
print("Project Structure:")
print("=" * 60)
print(f"Output directory: {OUTPUT_DIR}/")
for filename in os.listdir(OUTPUT_DIR):
    filepath = os.path.join(OUTPUT_DIR, filename)
    size_kb = os.path.getsize(filepath) / 1024
    print(f"  ✓ {filename} ({size_kb:.1f} KB)")

print("\n✓ Project ready for upload!")

---
## 6. Upload to MPLAB ML

### Authenticate with API Key

**Need an API key?** See the [creating-api-key.md](../../docs/getting-started/creating-api-key.md) guide.

In [ ]:
# Authenticate
api_key = getpass("Enter your MPLAB ML API Key: ")
client = Client(api_key=api_key)

print("✓ Connected to MPLAB ML!")

### Upload Project

In [ ]:
# Upload project
PROJECT_NAME = "Event_Labeled_Gesture_Tutorial"

print("=" * 70)
print(f"UPLOADING PROJECT: {PROJECT_NAME}")
print("=" * 70)
print(f"\nManifest: {manifest_filename}")
print(f"Upload directory: {OUTPUT_DIR}")

# Check if project already exists and delete it
print(f"\n🔍 Checking for existing project...")
try:
    projects_df = client.list_projects()
    if PROJECT_NAME in projects_df['Name'].values:
        print(f"   ⚠️  Project '{PROJECT_NAME}' already exists")
        print(f"   🗑️  Deleting existing project...")
        client.delete_project(PROJECT_NAME)
        print(f"   ✓ Deleted successfully")
    else:
        print(f"   ✓ No existing project found")
except Exception as e:
    print(f"   ℹ️  Could not check for existing project: {e}")

print(f"\n⏳ Upload in progress...")
print("   This may take 30-60 seconds...\n")

try:
    # Upload project with manifest
    result = client.upload_project_dcli(
        name=PROJECT_NAME,
        dcli_path=manifest_path,
        force=True
    )

    print("\n" + "=" * 70)
    print("✅ UPLOAD COMPLETED SUCCESSFULLY!")
    print("=" * 70)
    print(f"\nProject '{PROJECT_NAME}' is now in MPLAB ML!\n")
    print("💡 This is a tutorial example showing event-triggered labeling.")
    print("   You can safely delete it after exploring the workflow.\n")
    
except Exception as e:
    print("\n" + "=" * 70)
    print("❌ UPLOAD FAILED")
    print("=" * 70)
    print(f"\nError: {e}\n")
    print("Troubleshooting:")
    print("1. Try deleting the project manually in MPLAB ML web interface")
    print("2. Use a different project name (change PROJECT_NAME above)")
    print("3. Check your internet connection")
    print("4. Verify MPLAB ML credentials (API key)")
    print("5. Ensure manifest and CSV files are in same directory")
    print(f"6. Check files exist in: {OUTPUT_DIR}\n")
    raise

---
## Summary

### What We Accomplished:

1. ✅ **Loaded sensor data with trigger signal**
2. ✅ **Detected button press/release events** (rising/falling edges)
3. ✅ **Automatically labeled gesture segments** based on events
4. ✅ **Created manifest with segment definitions**
5. ✅ **Kept Trigger column for verification** in MPLAB ML
6. ✅ **Uploaded to MPLAB ML** for model training

### Key Concepts:

**Event-Triggered Labeling:**
- Hardware trigger provides ground truth timing
- Eliminates manual labeling
- Scalable to large datasets

**Rising/Falling Edges:**
- Rising edge (0→1) = Event START
- Falling edge (1→0) = Event END
- Segment between edges gets labeled

**Applications:**
- Gesture recognition (button during gesture)
- Activity tracking (start/stop activities)
- Anomaly detection (trigger on faults)
- Machine monitoring (trigger on operations)

---

## Next Steps

**In MPLAB ML:**
1. Open your project and verify the Trigger column aligns with labels
2. Create a query to select training data
3. Build feature extraction pipeline
4. Train and evaluate models
5. Generate Knowledge Pack for deployment

**💡 Pro Tip:** In MPLAB ML, you can plot the Trigger signal alongside your sensor data to visually confirm the automatic labeling worked correctly. Look for:
- Trigger signal transitions at segment boundaries
- Consistent timing between button press and gesture start
- No mislabeled idle periods

**Adapt for Your Hardware:**
- Button press → GPIO interrupt
- Switch state → Digital input
- PWM signal → Duty cycle threshold
- Sensor threshold → Comparator output

---

**🎉 Congratulations!** You've automated the labeling process with hardware triggers!